# AI Search & Rescue Robot — Cloud Model Fine-Tuning Pipeline
This Google Colab notebook fine-tunes **YOLO11n** or **YOLOv8n** on custom search & rescue objects or human postures, and exports the optimized `.pt` and `.onnx` models for deployment on the Raspberry Pi 5.

## 1. Verify GPU & Install Dependencies

In [ ]:
!nvidia-smi
!pip install -q ultralytics onnx onnxruntime

## 2. Prepare Dataset
Upload your `dataset.zip` (containing `images/` and `labels/` folders + `data.yaml`) or use a sample dataset.

In [ ]:
# If uploading a zip file:
# !unzip -q dataset.zip -d dataset/

# Or fine-tune on standard COCO classes (e.g. Person):
import yaml
print("Ready to train!")

## 3. Train YOLO11n Model

In [ ]:
from ultralytics import YOLO

# Load pre-trained base model
model = YOLO('yolo11n.pt')

# Train on dataset (replace 'coco8.yaml' with your 'dataset/data.yaml')
results = model.train(
    data='coco8.yaml',
    epochs=50,
    imgsz=640,
    batch=16,
    name='rescue_robot_yolo11n'
)

## 4. Export to ONNX (Raspberry Pi CPU Optimization)
ONNX models run at much higher FPS on Raspberry Pi OS.

In [ ]:
# Export to ONNX format
best_model = YOLO('runs/detect/rescue_robot_yolo11n/weights/best.pt')
onnx_path = best_model.export(format='onnx', imgsz=480)
print(f"Exported ONNX: {onnx_path}")

## 5. Download Trained Weights to Your PC
Download the weights and place them into `rescue_robot_pi/models/`.

In [ ]:
from google.colab import files
files.download('runs/detect/rescue_robot_yolo11n/weights/best.pt')
files.download('runs/detect/rescue_robot_yolo11n/weights/best.onnx')